In [1]:
import pandas as pd

flights    = pd.read_pickle("../data/processed/flights_clean.pkl")
payments   = pd.read_pickle("../data/processed/payments_clean.pkl")
bookings   = pd.read_pickle("../data/processed/bookings_analytics.pkl")     # masked version
passengers = pd.read_pickle("../data/processed/passengers_analytics.pkl")  # masked version

In [2]:
date_range = pd.date_range(
    start=flights["departure_time"].min().date(),
    end=flights["arrival_time"].max().date(),
    freq="D"
)
dim_date = pd.DataFrame({"date": date_range})
dim_date["date_id"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%B")
dim_date["day"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.strftime("%A")
dim_date["week_of_year"] = dim_date["date"].dt.isocalendar().week
dim_date["is_weekend"] = dim_date["date"].dt.dayofweek >= 5

In [3]:
dim_airline = pd.DataFrame({"airline": flights["airline"].unique()})
dim_airline["airline_id"] = range(1, len(dim_airline) + 1)

dim_route = flights[["source", "destination"]].drop_duplicates().reset_index(drop=True)
dim_route["route_id"] = range(1, len(dim_route) + 1)
dim_route["route_label"] = dim_route["source"] + " → " + dim_route["destination"]

In [4]:
dim_passenger = passengers[["passenger_id", "passenger_alias", "gender", "age_band"]].copy()

In [5]:
fact_flights = flights.merge(dim_airline, on="airline", how="left")
fact_flights = fact_flights.merge(dim_route, on=["source", "destination"], how="left")

fact_flights["dep_date_id"] = fact_flights["departure_time"].dt.strftime("%Y%m%d").astype(int)
fact_flights["arr_date_id"] = fact_flights["arrival_time"].dt.strftime("%Y%m%d").astype(int)

fact_flights = fact_flights[[
    "flight_id", "airline_id", "route_id",
    "departure_time", "arrival_time", "dep_date_id", "arr_date_id",
    "duration_minutes", "is_overnight", "is_corrupted_time"
]]

In [6]:
fact_bookings = bookings.merge(payments, on="booking_id", how="left")
fact_bookings = fact_bookings.merge(
    fact_flights[["flight_id"]].reset_index().rename(columns={"index": "flight_key"}),
    on="flight_id", how="left"
)

fact_bookings["booking_date_id"] = pd.to_datetime(fact_bookings["booking_date"]).dt.strftime("%Y%m%d").astype(int)

fact_bookings = fact_bookings[[
    "booking_id", "passenger_id", "flight_id", "booking_date_id",
    "status", "seat_number", "amount", "amount_is_missing", "payment_method"
]]

In [ ]:
print("fact_flights:", fact_flights.shape)
print("fact_bookings:", fact_bookings.shape)
print("dim_date:", dim_date.shape)
print("dim_airline:", dim_airline.shape)
print("dim_route:", dim_route.shape)
print("dim_passenger:", dim_passenger.shape)

print(fact_flights.isna().sum())
print(fact_bookings.isna().sum())

fact_flights: (1005, 10)
fact_bookings: (1363, 9)
dim_date: (5, 9)
dim_airline: (5, 2)
dim_route: (30, 4)
dim_passenger: (1039, 4)
flight_id            0
airline_id           0
route_id             0
departure_time       0
arrival_time         0
dep_date_id          0
arr_date_id          0
duration_minutes     1
is_overnight         0
is_corrupted_time    0
dtype: int64
booking_id             0
passenger_id           0
flight_id              0
booking_date_id        0
status                 0
seat_number            0
amount               441
amount_is_missing    363
payment_method       363
dtype: int64


In [8]:
import os
os.makedirs("../data/processed/model", exist_ok=True)

fact_flights.to_csv("../data/processed/model/fact_flights.csv", index=False)
fact_bookings.to_csv("../data/processed/model/fact_bookings.csv", index=False)
dim_date.to_csv("../data/processed/model/dim_date.csv", index=False)
dim_airline.to_csv("../data/processed/model/dim_airline.csv", index=False)
dim_route.to_csv("../data/processed/model/dim_route.csv", index=False)
dim_passenger.to_csv("../data/processed/model/dim_passenger.csv", index=False)